# Test: PBI Activity Events API
Simple test that uses a Service Principal (client credentials) to call the Power BI Admin Activity Events API once and print the response. No data is loaded anywhere.

In [ ]:
import requests
import json
from datetime import datetime, timedelta, timezone

# ---- Hardcoded values (TEST ONLY - do not commit real secrets) ----
TENANT_ID     = "<your-tenant-id>"
CLIENT_ID     = "<your-client-id>"
CLIENT_SECRET = "<your-client-secret>"

# How far back to query (Activity API allows up to 30 days, single window <= 24h)
WINDOW_HOURS = 1

In [ ]:
# 1. Get token via client credentials flow
token_url = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"
token_body = {
    "grant_type": "client_credentials",
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "scope": "https://analysis.windows.net/powerbi/api/.default",
}

print("POST", token_url)
print("Body (secret redacted):", {**token_body, "client_secret": "***REDACTED***"})
print(f"TENANT_ID len={len(TENANT_ID)}  CLIENT_ID len={len(CLIENT_ID)}  CLIENT_SECRET len={len(CLIENT_SECRET)}")

resp = requests.post(token_url, data=token_body)
print("\nStatus:", resp.status_code)
print("Response headers:", dict(resp.headers))
print("\n--- Raw response body ---")
print(resp.text)

try:
    body_json = resp.json()
    print("\n--- Parsed JSON ---")
    print(json.dumps(body_json, indent=2))
    if not resp.ok:
        print("\nERROR fields:")
        print("  error            :", body_json.get("error"))
        print("  error_description:", body_json.get("error_description"))
        print("  error_codes      :", body_json.get("error_codes"))
        print("  correlation_id   :", body_json.get("correlation_id"))
        print("  trace_id         :", body_json.get("trace_id"))
        print("  timestamp        :", body_json.get("timestamp"))
except Exception as e:
    print("Could not parse JSON:", e)

resp.raise_for_status()
access_token = resp.json()["access_token"]
print("\nToken acquired. Length:", len(access_token))


In [ ]:
# 2. Build time window (UTC, ISO with milliseconds, single-quoted as API requires)
end_dt   = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(hours=WINDOW_HOURS)

s_iso = start_dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
e_iso = end_dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

s_str = f"'{s_iso}'"
e_str = f"'{e_iso}'"

print("Start:", s_str)
print("End:  ", e_str)

In [ ]:
# 3. Call Activity Events API once and print result
filter_expr = "Activity eq 'EnableWorkspaceOutboundAccessProtection'"

url = (
    "https://api.powerbi.com/v1.0/myorg/admin/activityevents"
    f"?startDateTime={s_str}&endDateTime={e_str}&$filter={filter_expr}"
)

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}

print("GET", url)
r = requests.get(url, headers=headers)
print("Status:", r.status_code)
print("Response headers:", dict(r.headers))

try:
    data = r.json()
    print("\n--- Response body (JSON) ---")
    print(json.dumps(data, indent=2)[:5000])
    events = data.get("activityEventEntities") or data.get("value") or []
    print(f"\nEvent count: {len(events)}")
    print("continuationUri present:", bool(data.get("continuationUri")))
except Exception as e:
    print("Could not parse JSON:", e)
    print("Raw text:", r.text[:2000])